# Run Silver

- Generic Silver transformation notebook
- Called once per table for each task in Lakeflow jobs
- Input param table_name
- For each table:
    - Reads unprocessed batches from Bronze
    - Renamems columns
    - Cast to correct types
    - Deduplicates
    - Merge into silver table
    - Updates watermark

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from utils.logger import get_logger
from utils.silver_transform import get_missing_required, to_quarantine
from etl_config.constants_config import WATERMARK_TABLE
from etl_config.silver_config import SILVER_CONFIG

logger = get_logger("silver_transformation")

### Widget parameter

In [0]:
dbutils.widgets.text("table_name", "")
# table_name = dbutils.widgets.get("table_name")
table_name = "customers"

if not table_name or table_name not in SILVER_CONFIG:
    raise ValueError(f"Invalid or missing table name: {table_name}")

cfg = SILVER_CONFIG[table_name]
logger.info(f"Starting Silver processing for: {table_name}")

### Step 1: Find unprocessed Bronze batches for this table

In [0]:
watermark_row = spark.sql(f"""
    select max(ingested_at) as high_water_mark
    from {WATERMARK_TABLE}
    where table_name = '{table_name}'
    and status = 'SUCCESS'
""").collect()[0]

high_water_mark = watermark_row["high_water_mark"]

if high_water_mark is None:
    new_data_df = spark.table(cfg.source_table)
    logger.info(f"No watermark found for '{table_name}' — processing all Bronze data")
else:
    new_data_df = spark.table(cfg.source_table).filter(F.col("_ingested_at") > high_water_mark)
    logger.info(f"Watermark for '{table_name}': processing rows ingested after {high_water_mark}")

new_batch_ids = [
    row["_batch_id"]
    for row in new_data_df.select("_batch_id").distinct().collect()
]

if not new_batch_ids:
    logger.info(f"No new Bronze batches found for '{table_name}'. Nothing to do.")
    dbutils.notebook.exit("NO_NEW_BATCHES")

logger.info(f"Found {len(new_batch_ids)} new batch(es) to process: {new_batch_ids}")

input_count = new_data_df.count()

### Step 2: Rename columns

In [0]:
renamed_df = (
    new_data_df
        .withColumnsRenamed(cfg.column_mapping)
        .select(list(cfg.column_mapping.values()) + ["_source_file", "_batch_id", "_ingested_at"])
)

logger.info(f"Columns renamed for table: {table_name}")

### Step 3: Missing required columns

In [0]:
flagged_df = get_missing_required(renamed_df, cfg.required_columns)
missing_bad_df = flagged_df.filter(F.size("_missing_columns") > 0)
good_df = flagged_df.filter(F.size("_missing_columns") == 0).drop("_missing_columns")

missing_value_count = missing_bad_df.count()
to_quarantine(missing_bad_df, table_name, "MISSING_VALUE")

### Step 4: Cast to correct types

In [0]:
cast_df = good_df
for col_name, col_type in cfg.cast_columns.items():
    cast_df = (
        cast_df
            .withColumnRenamed(col_name, f"{col_name}_old")
            .withColumn(col_name, F.col(f"{col_name}_old").cast(col_type))
    )

# flag missing on cast columns
flagged_cast_df = get_missing_required(cast_df, cfg.required_columns)

# split into good and bad
good_df = flagged_cast_df.filter(F.size("_missing_columns") == 0).drop("_missing_columns")
cast_bad_df = flagged_cast_df.filter(F.size("_missing_columns") > 0)

for col_name in cfg.cast_columns.keys():
    cast_bad_df = (
        cast_bad_df
        .drop(col_name)
        .withColumnRenamed(f"{col_name}_old", col_name)
    )
    good_df = good_df.drop(f"{col_name}_old")

cast_failure_count = cast_bad_df.count()
to_quarantine(cast_bad_df, table_name, "CAST_FAILURE")

### Step 5: Deduplication

In [0]:
window_spec = Window.partitionBy(*cfg.key_columns).orderBy(F.col("_ingested_at").desc())

dedup_df = (
    good_df
    .withColumn("_row_num", F.row_number().over(window_spec))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

good_count = good_df.count()
dedup_count = dedup_df.count()
duplicate_count = good_count - dedup_count

if duplicate_count > 0:
    logger.warning(f"Found {duplicate_count} duplicate records in '{table_name}'.")

logger.info(f"{table_name}: {dedup_count} rows ready for merge")

### Step 6: Merge into SILVER

In [0]:
dedup_df.createOrReplaceTempView(f"stg_{table_name}")

merge_condition = " and ".join([f"target.{k} = source.{k}" for k in cfg.key_columns])

update_set = ", ".join([
    f"target.{col} = source.{col}"
    for col in cfg.cast_columns.keys()
    if col not in cfg.key_columns
])

insert_cols = ", ".join(cfg.cast_columns.keys())
insert_vals = ", ".join([f"source.{col}" for col in cfg.cast_columns.keys()])

spark.sql(f"""
    merge into {cfg.target_table} as target
    using stg_{table_name} as source
    on {merge_condition}
    when matched then update set {update_set}
    when not matched then insert ({insert_cols}) values ({insert_vals})
""")

logger.info(f"{table_name}: Merge complete into {cfg.target_table}")

### Step 7: Write in watermark table

In [0]:
rows_merged_df = (
    dedup_df.groupBy(F.col("_batch_id").alias("batch_id"))
    .count()
    .withColumnRenamed("count", "rows_merged")
)

# max _ingested_at po batch-u, from new_data_df (before transformations)
ingested_at_df = (
    new_data_df.groupBy(F.col("_batch_id").alias("batch_id"))
    .agg(F.max("_ingested_at").alias("ingested_at"))
)

new_batches_id_df = spark.createDataFrame([(b,) for b in new_batch_ids], ["batch_id"])

watermark_df = (
    new_batches_id_df
    .join(rows_merged_df, on="batch_id", how="left")
    .join(ingested_at_df, on="batch_id", how="left")
    .fillna({"rows_merged": 0})
    .withColumn("table_name", F.lit(table_name))
    .withColumn("processed_at", F.current_timestamp())
    .withColumn("status", F.lit("SUCCESS"))
    .select("table_name", "batch_id", "ingested_at", "processed_at", "rows_merged", "status")
)

watermark_df.write.format("delta").mode("append").saveAsTable(WATERMARK_TABLE)
logger.info(f"{table_name}: watermark updated for {len(new_batch_ids)} batch(es)")

### Step 8: Summary report

In [0]:
expected_total = missing_value_count + cast_failure_count + duplicate_count + dedup_count
row_count_mismatch = input_count - expected_total

logger.info(f"=== Silver Funnel Report: {table_name} ===")
logger.info(f"  Input from Bronze:         {input_count}")
logger.info(f"  - Quarantined (missing):   {missing_value_count}")
logger.info(f"  - Quarantined (cast fail): {cast_failure_count}")
logger.info(f"  - Duplicates dropped:      {duplicate_count}")
logger.info(f"  = Merged into Silver:      {dedup_count}")
if row_count_mismatch != 0:
    logger.warning(f"  ! Row count mismatch: {row_count_mismatch} rows")
logger.info("===========================================")